In [ ]:
import pandas as pd
import numpy as np
import os

# 1. Setup Paths for Data
RAW_DATA_PATH = '../data/raw/RBI_BankWise(2k16-26).xlsx' 
PROCESSED_DATA_PATH = '../data/processed/cleaned_fx_data.csv'

# 2. Load the data
df_raw = pd.read_excel(RAW_DATA_PATH)

# 3. Fix the Date Error
# 'dayfirst=True' tells Python that 31/12 is December 31st
df_raw['Date'] = pd.to_datetime(df_raw['Date'], dayfirst=True)

# 4. Sort and Clean Column Names
# We sort by date ascending so our 'Forward Fill' works correctly
df_raw = df_raw.sort_values('Date')

# Rename the long RBI names to simple ones for the team
column_mapping = {
    'USD (INR / 1 USD)': 'USD',
    'GBP (INR / 1 GBP)': 'GBP',
    'EUR (INR / 1 EUR)': 'EUR',
    'JPY (INR / 100 JPY)': 'JPY'
}
df_raw = df_raw.rename(columns=column_mapping)

# Keep only the columns we need
df_cleaned = df_raw[['Date', 'USD', 'GBP', 'EUR', 'JPY']].copy()
df_cleaned.set_index('Date', inplace=True)

# 5. Handle Weekends/Holidays
# This ensures every single day from 2016 to 2026 exists in our data
full_range = pd.date_range(start=df_cleaned.index.min(), end=df_cleaned.index.max(), freq='D')
df_final = df_cleaned.reindex(full_range)

# Fill gaps (Saturday/Sunday) with the Friday rate
df_final = df_final.ffill()

# 6. Calculate Volatility
currencies = ['USD', 'GBP', 'EUR', 'JPY']
for cur in currencies:
    # Daily % change
    df_final[f'{cur}_Return'] = df_final[cur].pct_change()
    # 30-day moving volatility
    df_final[f'{cur}_Volatility'] = df_final[f'{cur}_Return'].rolling(window=30).std()

# 7. Save for the team
os.makedirs('data/processed', exist_ok=True)
df_final.to_csv(PROCESSED_DATA_PATH)

print("Success! Date error resolved and Volatility calculated.")
print(f"Cleaned file saved at: {PROCESSED_DATA_PATH}")

Success! Date error resolved and Volatility calculated.
Cleaned file saved at: data/processed/cleaned_fx_data.csv
